## THC FeMoco compilation

In this notebook we provide an example of the CCZ cost model functionalities by attempting to reproduce results from section IV.C of [Lee et al. 2021](https://link.aps.org/doi/10.1103/PRXQuantum.2.030305) (surface code compilation of quantum phase estimation on the THC-encoded Li et al. FeMoco Hamiltonian).


### manual choice of distillation and data parameters

The physical costs can be computed by specifying manually the factories used for magic state distillation and the surface code parameters.

In [ ]:
from flagquantum.compiler._component_trace.resource_model import (
    GateCost,
    ModeledResources,
)
from flagquantum.qec.resources.ccz_to_t_factory import CCZToTFactory
from flagquantum.qec.resources.data_layout import RoutingDataLayout
from flagquantum.qec.resources.error_model import SurfaceCodeScheme
from flagquantum.qec.resources.parallel_factory import ParallelFactory
from flagquantum.qec.resources.physical_cost import PhysicalCostModel
from flagquantum.qec.resources.physical_parameters import PhysicalParameters

n_logical_gates = GateCost(toffoli=6665400000)  # Page 26.
n_algo_qubits = 696  # Figure 10.
logical = ModeledResources(n_logical_gates, n_algo_qubits)
factory = ParallelFactory(
    CCZToTFactory(level_one_distance=19, level_two_distance=31), n_factories=4
)
data_block = RoutingDataLayout(distance=31, routing_overhead=0.5)
cost = PhysicalCostModel(
    PhysicalParameters(1e-3, 1), data_block, factory, SurfaceCodeScheme.gidney_fowler()
).estimate(logical)
print(f"failure probability: {cost.failure_probability:.3%}")
print(f"wall time: {cost.duration_hours/24} days")  # Reference: 3 days.
print(
    f"footprint: {cost.physical_qubits*1e-6:.2f} million qubits"
)  # Reference: 4 million qubits.


### Grid search for distillation and data code distances

Alternatively, FlagQuantum provides a function to perform grid search over factories and data blocks, optimizing the cost (in terms of spacetime volume, total time or any other given ordering).

In [ ]:
from flagquantum.qec.resources.error_model import LogicalErrorModel

err_model = LogicalErrorModel(
    qec_scheme=SurfaceCodeScheme.gidney_fowler(), physical_error=1e-3
)


In [ ]:
from flagquantum.qec.resources.physical_search import (
    iter_ccz_factories,
    search_gidney_fowler_resources,
)

best_cost, best_factory, best_data_block = search_gidney_fowler_resources(
    logical,
    error_budget=1e-2,
    physical_error=err_model.physical_error,
    factories=iter_ccz_factories(n_factories=4),
    cost_function=lambda pc: pc.duration_hours,
)


In [ ]:
distillation_error = best_factory.factory_error(
    n_logical_gates, logical_error_model=err_model
)
data_error = best_data_block.storage_error(
    logical_qubits=n_algo_qubits,
    cycles=best_factory.n_cycles(n_logical_gates, logical_error_model=err_model),
    logical_error_model=err_model,
)
print(
    f"distillation error: {distillation_error:.3%}"
)  # Reference: 0.1% per 1e10 Toffolis.
print(f"data error: {data_error:.3%}")
print(f"total failure probability: {best_cost.failure_probability:.3%}")
print(f"wall time: {best_cost.duration_hours / 24} days")  # Reference: 3 days.
print(
    f"footprint: {best_cost.physical_qubits * 1e-6:.2f} million qubits"
)  # Reference: 4 million qubits.


The optimization result does not match the THC paper results exactly. 
The cost model used is similar, but the results from the THC paper include a detailed study of the lattice surgery layout. 
Here, instead, we just assume a 50% space overhead for routing and we compute the time as only limited by the CCZ generation rate.
The shorter runtime, at a cost of an increased number of physical qubits, might be due to this.

## Source and scope
Adapted from all seven cells of Qualtran v0.7.0, revision `bcb17bb25360ed68ce3bc084645db3e929799389`, blob `e82d0fee44ceb5f9cf2216ba929358e140003869`. Copyright 2023 Google LLC, Apache-2.0.
Both manual and grid-search costs follow the unmodified reference exactly. The 50% routing-overhead model differs from the cited paper’s detailed lattice-surgery layout. This tutorial consumes a declared logical workload. It does not compile or simulate the full FeMoco Hamiltonian. Primitive emission, measurement branches and workspace evidence remain separate. Circuit construction retains `circuit.append(op)`.
